# §39 step 1 — **ACCIDENT ingest**: download, 30 fps CFR, CLIP + V2-S caches, dedup vs TAD

Rule: `core/docs/final/KIM_RUN.md` **§39** (written before any number). Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. Needs a **GPU** (step 6). Nothing is trained here. **TAD frames are not needed on Drive**: the dedup reads `TAD_ncc` + the TAD label json; the contact sheet downloads only the few TAD images it draws from Kaggle (`nikanvasei/traffic-anomaly-dataset-tad`, the Lv 2021 release).

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, deps, helpers | no | 2 min |
| 4 | **download** `picekl/accident` with `kagglehub` (whole dataset ≈ 20 GB to VM disk), copy the 2,027 real mp4 (4.9 GB) + metadata to Drive once | no | 10-25 min |
| 5 | tests on the new code — **raises if red** | no | 1 min |
| 6 | **extract** 1,666 clips (cut ≥ one window), frames to `t_impact + 2 s`: CLIP on every CFR frame, V2-S on the s8 rows only, 100 clips per chunk, straight to Drive (resumable) | **yes (T4 ok)** | ~0.46 M CLIP frames + ~0.05 M V2-S clips (was 1.33 M): ≈ 10-20 min on a T4, decode is the floor |
| 7 | **dedup** vs the 100 TAD test videos (§39 D) | no | 5-10 min |
| 8 | contact sheet of flagged + just-below pairs -> **paste `REPORTS/accident_dedup.md` to Claude and look at the sheet** | no | 2 min |

**Kaggle login:** `kagglehub` downloads a public dataset without a key for most accounts. If it asks for one, add Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (Kaggle → Settings → API → Create New Token) and re-run step 4.

A re-run skips what is on Drive. Thesis is only read (TAD); everything new goes to `Thesis-Final/data/ACCIDENT/`, `Thesis-Final/cache/` and `Thesis-Final/outputs/kim_accident/`. **Stop after step 8**: step 2 (`k5dfi_acc.ipynb`) trains only after the dedup is read.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: TAD labels + TAD_ncc
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 39. K5dfiACC' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §39 -- re-upload kat-vad/'
for tool in ('accident_build', 'accident_extract', 'accident_dedup'):
    assert (REPO / f'core/tools/{tool}.py').is_file(), f'no core/tools/{tool}.py -- re-upload kat-vad/'
assert (REPO / 'core/splits/accident/ACCIDENT_MANIFEST.json').is_file(), 'no frozen ACCIDENT split -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

ENC      = constants.V5_ENCODER                                         # V2-S (K5dfi's motion stream)
ACC      = Path(DRIVE_F) / 'data' / constants.ACCIDENT_DATASET          # metadata + real_videos/ (Drive copy)
VIDEOS_D = ACC / constants.ACCIDENT_VIDEOS_DIRNAME
META_D   = ACC / constants.ACCIDENT_METADATA_FILENAME
CLIP_S1  = constants.CLIP_CACHE_DIR / constants.ACCIDENT_CLIP_S1_DATASET
VIDEO_S8 = constants.VIDEO_CACHE_DIR / ENC / f'{constants.ACCIDENT_VIDEO_S8_DATASET}_squash'   # V2-S at the s8 rows only
TAD      = constants.DATA_ROOT / constants.TAD_DATASET
TAD_NCC  = Path(DRIVE) / 'cache' / 'clip' / 'TAD_ncc'                   # plain CLIP, s8 (zs_lagovad's TAD input)
ROOT     = constants.OUTPUT_ROOT / constants.KIM_ACCIDENT_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/accident')
LOCAL.mkdir(parents=True, exist_ok=True)
SPLIT_MANIFEST = json.loads((constants.ACCIDENT_SPLITS_DIR / constants.ACCIDENT_SPLIT_MANIFEST_FILENAME).read_text())
for name, p in (('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME), ('TAD_ncc', TAD_NCC)):
    assert p.exists(), f'missing {name}: {p} -- paste to Claude'
print(f'code {REPO} | ACCIDENT -> {ACC} | caches {CLIP_S1}, {VIDEO_S8} | reports {REPORTS}')

In [ ]:
%%bash
pip install -q kagglehub "transformers==4.56.*" av einops faiss-cpu scikit-learn
python -c "import kagglehub, av, einops, faiss; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 6 needs one'
df -h /content | tail -1

In [ ]:
import hashlib  # noqa: E402

import numpy as np  # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    print(f'  staged {src.name}: {len(todo)} files in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def on_drive(folder, min_bytes=1024):
    """Ids of every finished ``.npy`` in ``folder`` from ONE listing (save_array = tmp + rename)."""
    if not folder.is_dir():
        return set()
    with os.scandir(folder) as entries:
        return {e.name[:-4] for e in entries if e.name.endswith('.npy') and e.stat().st_size >= min_bytes}


def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

## 4. Download (`kagglehub`) and copy the real clips to Drive once

The user's snippet, unchanged. The synthetic CARLA clips (13 GB) come with it and are **not used** (§39 A2); they are deleted from the VM after the copy. The Drive copy is the source of every later step, so a new VM never downloads again.

In [ ]:
meta_ids = None
if META_D.exists():
    import csv  # noqa: E402
    meta_ids = sorted(Path(r['path']).stem for r in csv.DictReader(META_D.open()))
have = {p.stem for p in VIDEOS_D.glob('*.mp4')} if VIDEOS_D.is_dir() else set()
if meta_ids is not None and set(meta_ids) <= have:
    print(f'Drive already holds {len(have)} real clips + metadata -- no download')
    SRC_VIDEOS = VIDEOS_D
else:
    import kagglehub  # noqa: E402

    # Download latest version
    path = kagglehub.dataset_download(constants.ACCIDENT_KAGGLE_HANDLE)
    print("Path to dataset files:", path)
    path = Path(path)
    src_meta, src_videos = path / constants.ACCIDENT_METADATA_FILENAME, path / constants.ACCIDENT_VIDEOS_DIRNAME
    assert src_meta.is_file() and src_videos.is_dir(), f'unexpected layout under {path}: {sorted(p.name for p in path.iterdir())}'
    ACC.mkdir(parents=True, exist_ok=True)
    for f in path.glob('*.csv'):
        shutil.copy2(f, ACC / f.name)
    for f in path.glob('*.yaml'):
        shutil.copy2(f, ACC / f.name)
    VIDEOS_D.mkdir(parents=True, exist_ok=True)
    todo = [p for p in sorted(src_videos.glob('*.mp4')) if not (VIDEOS_D / p.name).exists()]
    t0 = time.time()
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda p: shutil.copy2(p, VIDEOS_D / p.name), todo))
    print(f'copied {len(todo)} mp4 to Drive in {(time.time() - t0) / 60:.1f} min')
    shutil.rmtree(path / 'synthetic_videos', ignore_errors=True)   # not used (§39 A2)
    SRC_VIDEOS = src_videos                                         # read from VM disk this session (faster)
    flush_drive()

import csv  # noqa: E402
rows = {Path(r['path']).stem: r for r in csv.DictReader(META_D.open())}
ALL_IDS = sorted(rows)
assert len(ALL_IDS) == constants.ACCIDENT_REAL_VIDEOS, len(ALL_IDS)
assert sha1(META_D) == SPLIT_MANIFEST['metadata']['sha1'], \
    'metadata-real.csv differs from the one the frozen split was built on -- paste to Claude'
missing = [v for v in ALL_IDS if not (SRC_VIDEOS / f'{v}.mp4').exists()]
assert not missing, f'{len(missing)} clips have no mp4: {missing[:5]}'
size = sum((SRC_VIDEOS / f'{v}.mp4').stat().st_size for v in ALL_IDS) / 1e9
print(f'{len(ALL_IDS)} real clips ({size:.2f} GB) | metadata sha1 matches the frozen split | videos read from {SRC_VIDEOS}')
from core.tools.accident_build import windowable  # noqa: E402
EXTRACT_IDS = [v for v in ALL_IDS if windowable(rows[v])]     # a clip whose cut is < 1 window gives no window
print(f'to extract: {len(EXTRACT_IDS)} clips (skipped {len(ALL_IDS) - len(EXTRACT_IDS)} whose cut is shorter than one window)')

## 5. Tests on the new code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_accident.py',
     'core/tests/test_nexar_pipeline.py'])

## 6. Extract only what training reads → 30 fps CFR (by timestamp) → CLIP s1 + V2-S s8, straight to Drive

`core.tools.accident_extract`: frames only to `t_impact + 2 s` (the corpus cuts the rest), V2-S only at the s8 rows (`s1[::8]`, all that the bake reads), clips shorter than one window skipped. Measured on the metadata: **~459 k CFR frames → 50 k V2-S clips instead of 1.33 M** (≈ 23× less VideoMAE, ≈ 3× less CLIP); the cached rows are identical to a full extract followed by the cut (`test_decoding_to_the_cut_gives_the_full_decode_cut`, `test_motion_rows_at_a_stride_are_the_stride1_rows_subsampled`). 100 clips per chunk; a chunk on Drive is skipped, so re-run after a disconnect. **A T4 is enough.**

In [ ]:
CHUNK = 100
t_start = time.time()
done = on_drive(CLIP_S1) & on_drive(VIDEO_S8)
print(f'on Drive: {len(done)} / {len(EXTRACT_IDS)} clips have both caches')
chunks = [EXTRACT_IDS[i:i + CHUNK] for i in range(0, len(EXTRACT_IDS), CHUNK)]
for k, chunk in enumerate(chunks):
    todo = [v for v in chunk if v not in done]
    if not todo:
        continue
    print(f'=== chunk {k + 1}/{len(chunks)}: {len(todo)} clips')
    run([sys.executable, '-m', 'core.tools.accident_extract', '--videos-dir', SRC_VIDEOS, '--metadata', META_D,
         '--ids-file', write_ids(LOCAL / 'ids' / f'extract_{k:02d}.txt', todo), '--encoder', ENC,
         '--clip-out', CLIP_S1, '--video-out', VIDEO_S8,
         '--report', REPORTS / 'extract' / f'extract_{k:02d}_{int(time.time())}.json'], log=LOCAL / 'extract.log')
done = on_drive(CLIP_S1) & on_drive(VIDEO_S8)
left = [v for v in EXTRACT_IDS if v not in done]
print(f'extraction: {len(EXTRACT_IDS) - len(left)} / {len(EXTRACT_IDS)} ({(time.time() - t_start) / 60:.0f} min this session)')
assert not left, f'{len(left)} clips still to extract: re-run this cell'

In [ ]:
# Gate: V2-S rows == CLIP rows[::8]; CLIP rows == the cut (t_impact + 2 s) or the clip's end
def n_rows(path):
    return int(np.load(path, mmap_mode='r').shape[0])


with ThreadPoolExecutor(32) as pool:
    x_rows = dict(zip(EXTRACT_IDS, pool.map(lambda v: n_rows(CLIP_S1 / f'{v}.npy'), EXTRACT_IDS)))
    u_rows = dict(zip(EXTRACT_IDS, pool.map(lambda v: n_rows(VIDEO_S8 / f'{v}.npy'), EXTRACT_IDS)))
bad = [v for v in EXTRACT_IDS if u_rows[v] != -(-x_rows[v] // constants.FRAME_STRIDE)]
assert not bad, f'{len(bad)} clips: V2-S rows != CLIP rows[::8]: {bad[:5]}'
from core.tools.accident_build import cut_frames  # noqa: E402
full = {v: int(float(rows[v]['duration']) * constants.ACCIDENT_FPS) + 1 for v in EXTRACT_IDS}
gap = np.array([x_rows[v] - cut_frames(rows[v], full[v]) for v in EXTRACT_IDS])
print(f'V2-S == CLIP[::8] on {len(EXTRACT_IDS)} clips | CLIP rows - expected cut: median {np.median(gap):+.1f}, '
      f'|gap| > 30 frames (1 s) on {(np.abs(gap) > 30).sum()} clips, total {sum(x_rows.values()):,} CFR frames, '
      f'{sum(u_rows.values()):,} V2-S rows')
late = [v for v in EXTRACT_IDS if float(rows[v]['accident_time']) * constants.ACCIDENT_FPS >= x_rows[v]]
print(f'impact at or after the last CFR frame: {len(late)} clips {late[:5]} (they give no abnormal row)')

## 7. Dedup vs the 100 TAD test videos (§39 D)

τ = the 10th percentile of the same-source control: per upload with ≥ 2 clips, its best clip pair (same upload = same camera; the best pair only, since big uploads are multi-camera compilations). A clip whose best TAD match is ≥ τ is flagged and its whole source excluded. **STOP** if more than 10 % of clips would go.

In [ ]:
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(TAD_IDS) == constants.V2_TADZ_VIDEOS, len(TAD_IDS)
L_ACC = stage(CLIP_S1, LOCAL / 'stage' / 'acc_s1', EXTRACT_IDS)
L_TAD = stage(TAD_NCC, LOCAL / 'stage' / 'tad_ncc', TAD_IDS)
local = LOCAL / 'dedup'
run([sys.executable, '-m', 'core.tools.accident_dedup', '--acc-clip-s1-dir', L_ACC,
     '--acc-ids-file', write_ids(LOCAL / 'ids' / 'extracted.txt', EXTRACT_IDS), '--tad-clip-dir', L_TAD,
     '--tad-data-dir', TAD, '--out-dir', local])
publish(local, REPORTS)
dedup = json.loads((REPORTS / f'{constants.ACCIDENT_DEDUP_STEM}.json').read_text())
print((REPORTS / f'{constants.ACCIDENT_DEDUP_STEM}.md').read_text())

## 8. Contact sheet: each flagged pair (and the 20 just below τ) — ACCIDENT frame left, TAD frame right

Look at it. A flagged pair that is **not** the same camera can be kept by naming its source in `KEEP_SOURCES` in step 2's notebook (recorded in `KIM_RUN.md` §39 before training). A pair below τ that **is** the same camera: tell Claude.

In [ ]:
import av  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402

from core.tools.accident_extract import cfr_index  # noqa: E402


def acc_frame(clip_id, k):
    with av.open(str(SRC_VIDEOS / f'{clip_id}.mp4')) as c:
        frames = [(f.time, f.to_ndarray(format='rgb24')) for f in c.decode(c.streams.video[0])]
    idx = cfr_index(np.array([t for t, _ in frames]))
    return frames[int(idx[min(k, len(idx) - 1)])][1]


import kagglehub  # noqa: E402


def tad_frame(tad_id, k):
    # one TAD image, downloaded alone from Kaggle (TAD frames are not on Drive; only the sheet needs pixels)
    kind = 'normal' if tad_id.startswith('Normal') else 'abnormal'
    for j in (k, max(0, k - 8)):              # k = s8 row x 8; one step back if the folder ends early
        try:
            return plt.imread(kagglehub.dataset_download(constants.TAD_KAGGLE_HANDLE,
                                                         path=f'TAD/frames/{kind}/{tad_id}.mp4/{j}.jpg'))
        except Exception as exc:  # noqa: BLE001 -- a missing image must not stop the sheet
            print(f'TAD {tad_id} frame {j}: {exc}')
    return np.zeros((8, 8, 3))


MAX_DRAWN = 60                     # flagged pairs first, then the just-below ones
pairs = json.loads((REPORTS / f'{constants.ACCIDENT_DEDUP_STEM}_pairs.json').read_text())
if len(pairs) > MAX_DRAWN:
    print(f'{len(pairs)} pairs: drawing the first {MAX_DRAWN} (all listed in accident_dedup.md)')
    pairs = pairs[:MAX_DRAWN]
if not pairs:
    print('no pair to draw')
else:
    fig, axes = plt.subplots(len(pairs), 2, figsize=(9, 2.6 * len(pairs)), squeeze=False)
    for ax, p in zip(axes, pairs):
        ax[0].imshow(acc_frame(p['acc_id'], p['acc_frame']))
        ax[0].set_title(f"[{p['kind']}] {p['acc_id']} f{p['acc_frame']}  score {p['score']:.3f}", fontsize=8)
        ax[1].imshow(tad_frame(p['tad_id'], p['tad_frame']))
        ax[1].set_title(f"TAD {p['tad_id']} f{p['tad_frame']}", fontsize=8)
        for a in ax:
            a.axis('off')
    fig.tight_layout()
    sheet = LOCAL / 'dedup' / f'{constants.ACCIDENT_DEDUP_STEM}_sheet.png'
    fig.savefig(sheet, dpi=90)
    publish(sheet, REPORTS / sheet.name)
    plt.show()
print(f"status {dedup['status']} | tau {dedup['tau']:.4f} | excluded {len(dedup['excluded_clips'])} / {dedup['clips']} clips")
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/accident_dedup.md to Claude (and say what the sheet shows)')